## Runner

This example shows a dummy evaluation pipeline with `Runner`. We simulate a forecasting task
```
Time-series data → Sliding windows → Model inference → Metrics → Aggregated report
```
We need:

- `SlidingWindowDataset` to turn a time-series into samples
- `Runner` to orchestrate inference + metrics

In [1]:
import torch
import torch.nn as nn

In [2]:
from aiice.runner import Runner
from aiice.preprocess import SlidingWindowDataset

This model predicts the next step as: last observed timestep + 1.
It does not learn anything — just a deterministic rule. Useful to demonstrate pipeline mechanics

In [3]:
class DummyModel(nn.Module):
    def forward(self, x):
        # x shape: [history_len, channels, features] or similar
        # We take the last timestep and shift it
        return x[-1:] + 1.0

In [4]:
data = torch.arange(5 * 2 * 3, dtype=torch.float32).reshape(5, 2, 3)
data

tensor([[[ 0.,  1.,  2.],
         [ 3.,  4.,  5.]],

        [[ 6.,  7.,  8.],
         [ 9., 10., 11.]],

        [[12., 13., 14.],
         [15., 16., 17.]],

        [[18., 19., 20.],
         [21., 22., 23.]],

        [[24., 25., 26.],
         [27., 28., 29.]]])

We transform the sequence into supervised samples:

- History length = 2
- Forecast horizon = 1

In [5]:
dataset = SlidingWindowDataset(
    data=data,
    pre_history_len=2,
    forecast_len=1,
)

len(dataset)

3

In [6]:
x, y = dataset[0]
x.shape, y.shape

(torch.Size([2, 2, 3]), torch.Size([1, 2, 3]))

Runner ties everything together:

- loops over dataset
- calls model
- computes metrics
- aggregates statistics

We choose only MAE and MSE.

In [7]:
model = DummyModel()

runner = Runner(
    model=model,
    dataset=dataset,
    metrics=["mae", "mse"],
)

In [8]:
report = runner.run()
report

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 1627.80it/s]


{'mae': {'mean': 5.0, 'last': 5.0, 'count': 3, 'min': 5.0, 'max': 5.0},
 'mse': {'mean': 25.0, 'last': 25.0, 'count': 3, 'min': 25.0, 'max': 25.0}}